# 06 — QAT Inference Sweep

Loads ModelOpt-trained QAT weights from `checkpoints/qat/<PRECISION>_in<N>b/` and
evaluates each input-bit-width configuration on the held-out val split.

For each config we record `top1_acc`, `top5_acc`, `infer_ms_avg`, etc. and write a
`runs/qat/<run_id>/result.json` mirroring the schema produced by `runner.run_experiment`.

Inspired by `03_int8_cpu.ipynb` (PTQ sweep) but loads the model manually because
QAT graphs require `mto.restore_from_modelopt_state` before `load_state_dict`.


## Setup

In [ ]:
from pathlib import Path
import sys, json, time

sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("../src/qat_modelopt").resolve()))

import pandas as pd
import torch
import torch.nn as nn

from config import ExperimentConfig, set_seed
from data import build_runner_loaders
from eval import evaluate
from model import ResNet18
from quantize import restore_modelopt_state

pd.set_option("display.max_columns", 200)


## Configuration

Sweep over `INPUT_BITS_LIST`. For each bit-width we expect:

```
checkpoints/qat/<PRECISION>_in<N>b/
    qat_modelopt_best.pth
    qat_modelopt_best_mostate.pt
```

If a config's directory is missing, we skip and log it.


In [ ]:
REPO          = Path("..").resolve()
CKPT_ROOT     = REPO / "checkpoints" / "qat"
RUNS_ROOT     = REPO / "runs" / "qat"

PRECISION         = "int8"           # which QAT precision to evaluate
INPUT_BITS_LIST   = [1, 2, 4, 8]     # one run per bit-width
NUM_CLASSES       = 100
BATCH_SIZE        = 1
NUM_WORKERS       = 8
NUM_EVAL_BATCHES  = 500              # set to None for full val set
SEED              = 42
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

print(f"checkpoints  : {CKPT_ROOT}")
print(f"results dir  : {RUNS_ROOT}")
print(f"device       : {DEVICE}")


## Helpers

In [ ]:
def qat_run_id(precision: str, bits: int, batch_size: int, device: str) -> str:
    """Match the existing run_id convention used by runner.cfg.run_id()."""
    return f"resnet18_qat_modelopt_{precision}_in{bits}b_{device.split(':')[0]}_bs{batch_size}"


def load_qat_model(ckpt_dir: Path, num_classes: int, device: torch.device) -> nn.Module:
    """Rebuild a QAT'd ResNet-18 from (mostate, weights) on disk.

    Order matters:
      1. fresh ResNet-18 (architecture only)
      2. restore_modelopt_state -> inserts TensorQuantizer modules so keys match
      3. load_state_dict -> loads QAT-trained weights AND quantizer scales
    """
    pth_path = ckpt_dir / "qat_modelopt_best.pth"
    mo_path  = ckpt_dir / "qat_modelopt_best_mostate.pt"
    if not pth_path.exists() or not mo_path.exists():
        raise FileNotFoundError(f"Missing checkpoint pair in {ckpt_dir}")

    model = ResNet18(num_classes=num_classes, pretrained=False)
    restore_modelopt_state(model, str(mo_path))
    ckpt = torch.load(pth_path, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    return model.to(device).eval()


def make_payload(cfg: ExperimentConfig, run_id: str, ckpt_dir: Path, tracker) -> dict:
    """Mirror runner._make_payload but with QAT provenance fields added."""
    return {
        "status":  "ok",
        "run_id":  run_id,
        "system":  cfg.stamp(),
        "config":  {**cfg.to_dict(),
                    "qat_precision":      PRECISION,
                    "qat_checkpoint_dir": str(ckpt_dir)},
        "results": tracker.summary(),
        "error":   None,
    }


def save_result(payload: dict, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "result.json"
    path.write_text(json.dumps(payload, indent=2, sort_keys=True))
    return path


## Run inference for each input-bit-width

In [ ]:
records = []

for bits in INPUT_BITS_LIST:
    run_tag  = f"{PRECISION}_in{bits}b"
    ckpt_dir = CKPT_ROOT / run_tag
    run_id   = qat_run_id(PRECISION, bits, BATCH_SIZE, DEVICE)

    print(f"\n=== {run_id} ===")

    if not (ckpt_dir / "qat_modelopt_best.pth").exists():
        print(f"  [skip] no checkpoint at {ckpt_dir}")
        continue

    cfg = ExperimentConfig(
        backend          = "pytorch",
        device           = DEVICE,
        model_precision  = "fp32",      # framework-side dtype; fake-quant lives inside the graph
        batch_size       = BATCH_SIZE,
        num_workers      = NUM_WORKERS,
        input_quant_bits = bits,
        num_classes      = NUM_CLASSES,
        num_eval_batches = NUM_EVAL_BATCHES,
        seed             = SEED,
    )
    cfg.validate()
    set_seed(cfg)

    device  = torch.device(cfg.device)
    model   = load_qat_model(ckpt_dir, num_classes=NUM_CLASSES, device=device)
    _, val_loader = build_runner_loaders(cfg)

    t0      = time.perf_counter()
    tracker = evaluate(model, val_loader, cfg, criterion=nn.CrossEntropyLoss())

    payload = make_payload(cfg, run_id, ckpt_dir, tracker)
    payload["total_eval_time_sec"] = round(time.perf_counter() - t0, 3)
    saved = save_result(payload, RUNS_ROOT / run_id)
    print(f"  saved -> {saved}")

    records.append(payload)


## Summary

In [ ]:
rows = []
for r in records:
    res = r["results"]
    rows.append({
        "run_id":         r["run_id"],
        "input_bits":     r["config"]["input_quant_bits"],
        "qat_precision":  r["config"]["qat_precision"],
        "top1_acc":       res["top1_acc"],
        "top5_acc":       res["top5_acc"],
        "infer_ms_avg":   res["infer_ms_avg"],
        "throughput_sps": res["throughput_infer_sps"],
        "total_samples":  res["total_samples"],
    })

df = pd.DataFrame(rows).sort_values("input_bits").reset_index(drop=True)
df
